# PHLOP Cond-T: No-Video Ablation

Run this on a GPU box. **No retraining** — loads base and/or existing LoRA checkpoints and evaluates with blank video + a text-only prompt.

**Outputs** (under `RESULTS_DIR`):
- `text_only_predictions_{tag}.json` — per-question preds
- `text_only_summary_{tag}.json` — overall / by difficulty / by question_type

Workshop minimum: one model × one curriculum (`full` or `easy_medium`) × both physics settings. Full grid optional.

## 0. Config — edit this cell

In [ ]:
import os
from pathlib import Path

# Which model family to run this session
MODEL_FAMILY = "smolvlm"  # "smolvlm" | "internvl3" | "qwen2vl"

# HuggingFace dataset
REPO_ID = os.environ.get("PHLOP_REPO_ID", "zimmari-ai/phlop")
HF_TOKEN = os.environ.get("HF_TOKEN", True)  # True = use cached login

CAMERA_MODE = "static"
SPLIT = "test"

# Checkpoint roots from your previous fine-tunes (adjust to your GPU machine paths)
CKPT_ROOTS = {
    "smolvlm": os.environ.get("SMOLVLM_CKPT_ROOT", "./smolvlm_physics_full"),
    "internvl3": os.environ.get("INTERNVL3_CKPT_ROOT", "./internvl3_physics_full"),
    "qwen2vl": os.environ.get("QWEN2VL_CKPT_ROOT", "./qwen2vl_physics_full"),
}

# Curricula to evaluate. "base" = non-finetuned hub weights.
# Directory layout expected: {CKPT_ROOT}/{config_name}/  OR  {CKPT_ROOT}/{model}_{config}_{physics}/
# Set USE_PHYSICS_IN_CKPT_NAME if your dirs include _with_physics / _no_physics suffixes.
CONFIGS = ["base", "full"]  # workshop min: ["base", "full"] or ["base", "easy_medium"]
PHYSICS_SETTINGS = [False, True]  # no_physics, with_physics
USE_PHYSICS_IN_CKPT_NAME = True  # set False if ckpt dirs are just easy/full/...

MAX_SAMPLES = None  # e.g. 200 for a smoke test; None = full test set
NUM_FRAMES = 8  # blank frames fed to the vision tower (keep small for speed)
BATCH_SIZE = 2  # SmolVLM only

RESULTS_DIR = Path(os.environ.get("PHLOP_TEXT_ONLY_OUT", "./results_text_only"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BLANK_VIDEO = RESULTS_DIR / "blank_8f.mp4"

print("MODEL_FAMILY:", MODEL_FAMILY)
print("CKPT_ROOT:", CKPT_ROOTS[MODEL_FAMILY])
print("CONFIGS:", CONFIGS)
print("PHYSICS_SETTINGS:", PHYSICS_SETTINGS)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("RESULTS_DIR:", RESULTS_DIR.resolve())

## 1. Imports + load PHLOP test split

In [ ]:
import sys
import gc
import json
import torch

NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    # allow running from repo root
    if (NB_DIR / "notebooks").is_dir():
        NB_DIR = NB_DIR / "notebooks"
sys.path.insert(0, str(NB_DIR))

from phlop_eval_common import (
    load_phlop_splits,
    FINE_TUNE_CONFIGS,
    compute_metrics,
    print_metrics,
)
from text_only_eval_helpers import (
    TextOnlySmolWrapper,
    TextOnlyInternVLWrapper,
    TextOnlyQwenWrapper,
    ensure_blank_mp4,
    score_and_summarize,
    save_run,
)
from text_only_leak_protocol import VIDEO_OPTIONAL_HIGH

splits = load_phlop_splits(
    REPO_ID,
    token=HF_TOKEN,
    extract_root=os.environ.get("PHLOP_EXTRACT_ROOT"),
)
assert SPLIT in splits, f"missing split {SPLIT}; have {list(splits.keys())}"
print("splits:", {k: len(v) for k, v in splits.items()})
print("High-leak types:", sorted(VIDEO_OPTIONAL_HIGH))

## 2. Resolve checkpoint paths

Edit `resolve_ckpt` if your directory naming differs.

In [ ]:
BASE_IDS = {
    "smolvlm": "HuggingFaceTB/SmolVLM2-2.2B-Instruct",
    "internvl3": "OpenGVLab/InternVL3-2B",  # confirm against internvl3_eval.INTERNVL3_MODEL_ID
    "qwen2vl": "Qwen/Qwen2-VL-2B-Instruct",
}

# Prefer IDs from the eval modules when available
try:
    import smol_eval as _sm
    BASE_IDS["smolvlm"] = getattr(_sm, "SMOLVLM_MODEL_ID", BASE_IDS["smolvlm"])
except Exception:
    pass
try:
    import internvl3_eval as _iv
    BASE_IDS["internvl3"] = getattr(_iv, "INTERNVL3_MODEL_ID", BASE_IDS["internvl3"])
except Exception:
    pass
try:
    import qwen2vl_eval as _qw
    BASE_IDS["qwen2vl"] = getattr(_qw, "QWEN2_VL_MODEL_ID", BASE_IDS["qwen2vl"])
except Exception:
    pass


def resolve_ckpt(family: str, config_name: str, use_physics: bool) -> str | None:
    """Return hub id or local dir. None if missing (skip)."""
    if config_name == "base":
        return BASE_IDS[family]
    root = Path(CKPT_ROOTS[family])
    physics_tag = "with_physics" if use_physics else "no_physics"
    candidates = []
    if USE_PHYSICS_IN_CKPT_NAME:
        candidates += [
            root / f"{family}_{config_name}_{physics_tag}",
            root / f"{config_name}_{physics_tag}",
            root / physics_tag / config_name,
        ]
    candidates += [
        root / config_name,
        root / f"{family}_{config_name}",
    ]
    for c in candidates:
        if c.is_dir():
            return str(c)
    print(f"  [skip] no checkpoint for {family}/{config_name}/{physics_tag}. tried:")
    for c in candidates:
        print(f"         {c}")
    return None


# Preview what will run
plan = []
for cfg in CONFIGS:
    for phys in PHYSICS_SETTINGS:
        ckpt = resolve_ckpt(MODEL_FAMILY, cfg, phys)
        plan.append((cfg, phys, ckpt))
        print(f"{cfg:12} physics={phys!s:5} → {ckpt}")

## 3. Run Cond-T for the selected family

Only the cell matching `MODEL_FAMILY` needs to succeed; others can be skipped.

In [ ]:
def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


all_summaries = {}

### 3a. SmolVLM

In [ ]:
if MODEL_FAMILY == "smolvlm":
    from transformers import AutoProcessor, AutoModelForImageTextToText
    from peft import PeftModel
    from smol_eval import PHLOPValDataset, collect_predictions_smolvlm, _get_device, _best_dtype

    device = _get_device()
    dtype = _best_dtype(device)
    processor = AutoProcessor.from_pretrained(BASE_IDS["smolvlm"])
    processor.tokenizer.padding_side = "left"

    for cfg, use_physics, ckpt in plan:
        if ckpt is None:
            continue
        physics_tag = "with_physics" if use_physics else "no_physics"
        tag = f"smolvlm_{cfg}_{physics_tag}_{CAMERA_MODE}_text_only"
        print(f"\n=== {tag} ===")

        base_ds = PHLOPValDataset(
            splits[SPLIT],
            camera_mode=CAMERA_MODE,
            num_frames=NUM_FRAMES,
            use_physics=use_physics,
        )
        ds = TextOnlySmolWrapper(base_ds, num_frames=NUM_FRAMES)

        if cfg == "base":
            model = AutoModelForImageTextToText.from_pretrained(ckpt, dtype=dtype)
        else:
            # Prefer PEFT load; fall back to direct from_pretrained (matches smol_eval comparison)
            try:
                base_model = AutoModelForImageTextToText.from_pretrained(
                    BASE_IDS["smolvlm"], dtype=dtype
                )
                model = PeftModel.from_pretrained(base_model, ckpt)
            except Exception as e:
                print(f"  PEFT load failed ({e}); trying direct from_pretrained({ckpt})")
                model = AutoModelForImageTextToText.from_pretrained(ckpt, dtype=dtype)
        model = model.to(device).eval()

        results = collect_predictions_smolvlm(
            model, processor, ds,
            max_samples=MAX_SAMPLES,
            device=device,
            batch_size=BATCH_SIZE,
        )
        for r in results:
            r["model"] = f"smolvlm_{cfg}"
            r["use_physics"] = use_physics
            r["text_only"] = True
            r["split"] = SPLIT

        summary = score_and_summarize(results)
        paths = save_run(results, summary, RESULTS_DIR, tag)
        all_summaries[tag] = {"summary": summary, **paths}

        del model, results
        _free()
else:
    print("Skipping SmolVLM (MODEL_FAMILY != smolvlm)")

### 3b. InternVL3

In [ ]:
if MODEL_FAMILY == "internvl3":
    from transformers import AutoTokenizer, AutoModel
    from peft import PeftModel
    import internvl3_eval as iv

    device = iv._get_device() if hasattr(iv, "_get_device") else (
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    dtype = iv._best_dtype(device) if hasattr(iv, "_best_dtype") else torch.bfloat16
    model_id = BASE_IDS["internvl3"]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    for cfg, use_physics, ckpt in plan:
        if ckpt is None:
            continue
        physics_tag = "with_physics" if use_physics else "no_physics"
        tag = f"internvl3_{cfg}_{physics_tag}_{CAMERA_MODE}_text_only"
        print(f"\n=== {tag} ===")

        base_ds = iv.PHLOPValDataset(
            splits[SPLIT],
            camera_mode=CAMERA_MODE,
            num_frames=NUM_FRAMES,
            use_physics=use_physics,
        )
        ds = TextOnlyInternVLWrapper(base_ds, num_frames=NUM_FRAMES)

        if cfg == "base":
            model = AutoModel.from_pretrained(
                ckpt, torch_dtype=dtype, trust_remote_code=True
            )
        else:
            base_model = AutoModel.from_pretrained(
                model_id, torch_dtype=dtype, trust_remote_code=True
            )
            model = PeftModel.from_pretrained(base_model, ckpt)
        model = model.to(device).eval()

        results = iv.collect_predictions_internvl3(
            model, tokenizer, ds,
            max_samples=MAX_SAMPLES,
            device=device,
            num_frames=NUM_FRAMES,
        )
        for r in results:
            r["model"] = f"internvl3_{cfg}"
            r["use_physics"] = use_physics
            r["text_only"] = True
            r["split"] = SPLIT

        summary = score_and_summarize(results)
        paths = save_run(results, summary, RESULTS_DIR, tag)
        all_summaries[tag] = {"summary": summary, **paths}

        del model, results
        _free()
else:
    print("Skipping InternVL3 (MODEL_FAMILY != internvl3)")

### 3c. Qwen2-VL

In [ ]:
if MODEL_FAMILY == "qwen2vl":
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
    from peft import PeftModel
    import qwen2vl_eval as qw

    blank_path = ensure_blank_mp4(BLANK_VIDEO, num_frames=NUM_FRAMES)
    print("Blank video:", blank_path)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model_id = BASE_IDS["qwen2vl"]
    processor = AutoProcessor.from_pretrained(model_id)

    for cfg, use_physics, ckpt in plan:
        if ckpt is None:
            continue
        physics_tag = "with_physics" if use_physics else "no_physics"
        tag = f"qwen2vl_{cfg}_{physics_tag}_{CAMERA_MODE}_text_only"
        print(f"\n=== {tag} ===")

        base_ds = qw.PHLOPValDataset(
            splits[SPLIT],
            camera_mode=CAMERA_MODE,
            use_physics=use_physics,
        )
        ds = TextOnlyQwenWrapper(base_ds, blank_video_path=blank_path)

        if cfg == "base":
            model = Qwen2VLForConditionalGeneration.from_pretrained(ckpt, torch_dtype=dtype)
        else:
            base_model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_id, torch_dtype=dtype
            )
            model = PeftModel.from_pretrained(base_model, ckpt)
        model = model.to(device).eval()

        results = qw.collect_predictions_qwen2vl(
            model, processor, ds,
            max_samples=MAX_SAMPLES,
            device=device,
        )
        for r in results:
            r["model"] = f"qwen2vl_{cfg}"
            r["use_physics"] = use_physics
            r["text_only"] = True
            r["split"] = SPLIT

        summary = score_and_summarize(results)
        paths = save_run(results, summary, RESULTS_DIR, tag)
        all_summaries[tag] = {"summary": summary, **paths}

        del model, results
        _free()
else:
    print("Skipping Qwen2-VL (MODEL_FAMILY != qwen2vl)")

## 4. Master summary table

What to put in the paper:
- **Cond-T full** ≈ upper bound on non-visual score
- **Cond-T high-leak types** should be high if leakage is real
- **Cond-T on video-required** should be near chance

In [ ]:
rows = []
for tag, payload in all_summaries.items():
    s = payload["summary"]
    rows.append({
        "tag": tag,
        "T_full_acc": s["cond_T_full"]["acc"],
        "T_full_n": s["cond_T_full"]["n"],
        "T_high_leak_acc": s["cond_T_high_leak_types_only"]["acc"],
        "T_high_leak_n": s["cond_T_high_leak_types_only"]["n"],
        "T_video_req_acc": s["cond_T_on_video_required_templates"]["acc"],
        "T_video_req_n": s["cond_T_on_video_required_templates"]["n"],
    })

master_path = RESULTS_DIR / f"text_only_master_{MODEL_FAMILY}.json"
with open(master_path, "w") as f:
    json.dump({"runs": all_summaries, "table": rows}, f, indent=2, default=str)

print(f"Wrote {master_path}\n")
for r in rows:
    print(
        f"{r['tag']}\n"
        f"  Cond-T full:       {r['T_full_acc']:.3f} (n={r['T_full_n']})\n"
        f"  Cond-T high-leak:  {r['T_high_leak_acc']:.3f} (n={r['T_high_leak_n']})\n"
        f"  Cond-T video-req:  {r['T_video_req_acc']:.3f} (n={r['T_video_req_n']})\n"
    )

## Smoke test tip

Before a full test pass, set `MAX_SAMPLES = 100` and `CONFIGS = ["base"]`, `PHYSICS_SETTINGS = [False]`. Confirm JSON files appear under `RESULTS_DIR`, then set `MAX_SAMPLES = None` and expand the grid.

Repeat the notebook with `MODEL_FAMILY` set to each of `smolvlm`, `internvl3`, `qwen2vl` (or only the one you need for the workshop).